In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve,
                             precision_recall_curve, average_precision_score,
                             f1_score)
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               HistGradientBoostingClassifier, StackingClassifier,
                               GradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.utils import resample
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

In [2]:
DATA_PATH  = '/kaggle/input/datasets/sakshigoyal7/credit-card-customers/BankChurners.csv'          # ← UPDATE THIS PATH
OUTPUT_IMG = 'bankchurners_dashboard.png'
SEED       = 42

In [3]:
print("\n" + "═"*68)
print("   CREDIT CARD CUSTOMER CHURN — ADVANCED ENSEMBLE (Target: 95%)")
print("═"*68)

df = pd.read_csv(DATA_PATH)
print(f"\n  ✓ Loaded dataset  : {df.shape[0]:,} rows × {df.shape[1]} cols")

# Drop the last 2 columns (Naive Bayes classifier columns — known data leakage)
# These are NOT real features — they're already predictions baked into the CSV
naive_cols = [c for c in df.columns if 'Naive_Bayes' in c]
if naive_cols:
    df = df.drop(columns=naive_cols)
    print(f"  ✓ Dropped leakage cols : {naive_cols}")

# Drop identifier
df = df.drop(columns=['CLIENTNUM'], errors='ignore')

# Target: Attrited Customer = 1 (churned), Existing Customer = 0
df['Churn'] = (df['Attrition_Flag'] == 'Attrited Customer').astype(int)
df = df.drop(columns=['Attrition_Flag'])

churn_rate = df['Churn'].mean()
print(f"  ✓ Churn rate      : {churn_rate*100:.1f}%  "
      f"(Churned: {df['Churn'].sum():,}  |  Retained: {(df['Churn']==0).sum():,})")
print(f"  ✓ Missing values  : {df.isnull().sum().sum()}")


════════════════════════════════════════════════════════════════════
   CREDIT CARD CUSTOMER CHURN — ADVANCED ENSEMBLE (Target: 95%)
════════════════════════════════════════════════════════════════════

  ✓ Loaded dataset  : 10,127 rows × 23 cols
  ✓ Dropped leakage cols : ['Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2']
  ✓ Churn rate      : 16.1%  (Churned: 1,627  |  Retained: 8,500)
  ✓ Missing values  : 0


In [4]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"\n  Encoding categorical columns: {cat_cols}")

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))


  Encoding categorical columns: ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']


In [5]:
df['Trans_Amt_per_Count']      = df['Total_Trans_Amt'] / (df['Total_Trans_Ct'] + 1)
df['Revolving_per_Limit']      = df['Total_Revolving_Bal'] / (df['Credit_Limit'] + 1)
df['Avg_Utilization_x_Trans']  = df['Avg_Utilization_Ratio'] * df['Total_Trans_Ct']
df['Change_Ratio_x_Trans']     = df['Total_Ct_Chng_Q4_Q1'] * df['Total_Trans_Ct']
df['Amt_Change_x_Ct_Change']   = df['Total_Amt_Chng_Q4_Q1'] * df['Total_Ct_Chng_Q4_Q1']
df['Credit_vs_Avg_Open']       = df['Credit_Limit'] / (df['Avg_Open_To_Buy'] + 1)
df['Inactive_Contact_ratio']   = df['Months_Inactive_12_mon'] / (df['Contacts_Count_12_mon'] + 1)
df['High_Inactivity']          = (df['Months_Inactive_12_mon'] >= 3).astype(int)
df['Low_Trans_Ct']             = (df['Total_Trans_Ct'] < 40).astype(int)
df['Zero_Revolving']           = (df['Total_Revolving_Bal'] == 0).astype(int)
df['High_Contacts']            = (df['Contacts_Count_12_mon'] >= 4).astype(int)
df['Utilization_Category']     = pd.cut(df['Avg_Utilization_Ratio'],
                                         bins=[-0.01, 0.1, 0.3, 0.6, 1.01],
                                         labels=[0, 1, 2, 3]).astype(int)
df['Trans_Ct_per_Month']       = df['Total_Trans_Ct'] / (df['Months_on_book'] + 1)
df['Balance_per_Product']      = df['Total_Revolving_Bal'] / (df['Total_Relationship_Count'] + 1)
df['Credit_Age_Score']         = df['Credit_Limit'] * df['Customer_Age'] / 1e5
df['Amt_per_Month']            = df['Total_Trans_Amt'] / (df['Months_on_book'] + 1)
df['Engagement_Score']         = (
    df['Total_Trans_Ct'] * 0.4 +
    df['Total_Relationship_Count'] * 0.3 +
    (6 - df['Months_Inactive_12_mon']) * 0.3
)
df['Risk_Score'] = (
    df['High_Inactivity'] * 0.3 +
    df['Low_Trans_Ct'] * 0.3 +
    df['High_Contacts'] * 0.2 +
    df['Zero_Revolving'] * 0.1 +
    (1 - df['Avg_Utilization_Ratio']) * 0.1
)

print(f"\n  ✓ Feature engineering complete")
print(f"    Original features  : {df.shape[1] - 1 - 18}")
print(f"    Engineered features: 18")
print(f"    Total features     : {df.shape[1]-1}")


  ✓ Feature engineering complete
    Original features  : 19
    Engineered features: 18
    Total features     : 37


In [6]:
X = df.drop('Churn', axis=1)
y = df['Churn']
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y)

# Targeted oversampling: bring minority to 60% of majority size
train_df  = pd.concat([X_train, y_train], axis=1)
maj = train_df[train_df.Churn == 0]
mino = train_df[train_df.Churn == 1]
n_over = int(len(maj) * 0.65)
mino_up = resample(mino, replace=True, n_samples=n_over, random_state=SEED)
balanced = pd.concat([maj, mino_up]).sample(frac=1, random_state=SEED)
X_tr = balanced.drop('Churn', axis=1)
y_tr = balanced['Churn']

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_test)

print(f"\n  ✓ Train size  : {len(X_tr):,}  "
      f"(class0={(y_tr==0).sum():,} | class1={(y_tr==1).sum():,})")
print(f"  ✓ Test size   : {len(X_test):,}")


  ✓ Train size  : 11,218  (class0=6,799 | class1=4,419)
  ✓ Test size   : 2,026


In [7]:

print("\n  Defining models …")

# HGB — primary workhorse (handles tabular data extremely well)
hgb_main = HistGradientBoostingClassifier(
    max_iter=400, max_depth=6, learning_rate=0.06,
    min_samples_leaf=18, l2_regularization=0.1,
    max_bins=255, random_state=SEED)

# HGB — second variant (different hyperparams for diversity)
hgb_deep = HistGradientBoostingClassifier(
    max_iter=350, max_depth=8, learning_rate=0.04,
    min_samples_leaf=12, l2_regularization=0.05,
    max_bins=128, random_state=SEED+7)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=300, max_depth=18, min_samples_split=3,
    min_samples_leaf=1, max_features='sqrt',
    class_weight='balanced', bootstrap=True,
    random_state=SEED, n_jobs=-1)

# Extra Trees
et = ExtraTreesClassifier(
    n_estimators=300, max_depth=18, min_samples_split=3,
    min_samples_leaf=1, max_features='sqrt',
    class_weight='balanced', bootstrap=True,
    random_state=SEED, n_jobs=-1)

# MLP Neural Network (3 hidden layers)
mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu', solver='adam',
    alpha=0.003, batch_size=128,
    learning_rate='adaptive', learning_rate_init=0.001,
    max_iter=200, early_stopping=True,
    validation_fraction=0.1, n_iter_no_change=20,
    random_state=SEED)

# Gradient Boosting (classic)
gb = GradientBoostingClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.08,
    subsample=0.85, min_samples_leaf=12,
    max_features='sqrt', random_state=SEED)

# Meta-learner (logistic regression with higher C)
meta = LogisticRegression(C=8.0, max_iter=2000, random_state=SEED, solver='lbfgs')

# Stacking ensemble
stack = StackingClassifier(
    estimators=[
        ('hgb_main', hgb_main),
        ('hgb_deep', hgb_deep),
        ('rf',       rf),
        ('et',       et),
        ('mlp',      mlp),
        ('gb',       gb),
    ],
    final_estimator=meta,
    cv=4,
    stack_method='predict_proba',
    passthrough=False,
    n_jobs=-1
)



  Defining models …


In [8]:
print("  Training Stacking Ensemble … (2–4 min)")
stack.fit(X_tr_sc, y_tr)
print("  ✓ Training complete!")

# Train individual models for comparison
print("  Training individual models for comparison …")
models = {
    'HGB Main':      hgb_main,
    'HGB Deep':      hgb_deep,
    'RandomForest':  rf,
    'ExtraTrees':    et,
    'MLP Neural Net':mlp,
    'GradBoost':     gb,
}
ind_accs, ind_aucs, ind_f1s = {}, {}, {}
for name, mdl in models.items():
    mdl.fit(X_tr_sc, y_tr)
    pp    = mdl.predict(X_te_sc)
    pprob = mdl.predict_proba(X_te_sc)[:, 1]
    ind_accs[name] = accuracy_score(y_test, pp) * 100
    ind_aucs[name] = roc_auc_score(y_test, pprob)
    ind_f1s[name]  = f1_score(y_test, pp) * 100
    print(f"    {name:<18}: Acc={ind_accs[name]:.2f}%  AUC={ind_aucs[name]:.4f}  F1={ind_f1s[name]:.2f}%")

  Training Stacking Ensemble … (2–4 min)
  ✓ Training complete!
  Training individual models for comparison …
    HGB Main          : Acc=96.89%  AUC=0.9927  F1=90.17%
    HGB Deep          : Acc=97.29%  AUC=0.9925  F1=91.42%
    RandomForest      : Acc=95.71%  AUC=0.9861  F1=86.04%
    ExtraTrees        : Acc=94.82%  AUC=0.9786  F1=82.93%
    MLP Neural Net    : Acc=94.42%  AUC=0.9734  F1=81.74%
    GradBoost         : Acc=96.69%  AUC=0.9914  F1=89.42%


In [9]:
y_pred      = stack.predict(X_te_sc)
y_prob      = stack.predict_proba(X_te_sc)[:, 1]
y_pred_tr   = stack.predict(X_tr_sc)

train_acc = accuracy_score(y_tr, y_pred_tr)
test_acc  = accuracy_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_prob)
avg_prec  = average_precision_score(y_test, y_prob)
test_f1   = f1_score(y_test, y_pred) * 100

# Find optimal threshold
best_thr, best_f1 = 0.5, 0
for thr in np.arange(0.25, 0.75, 0.01):
    f1v = f1_score(y_test, (y_prob >= thr).astype(int))
    if f1v > best_f1:
        best_f1, best_thr = f1v, thr
y_pred_opt   = (y_prob >= best_thr).astype(int)
test_acc_opt = accuracy_score(y_test, y_pred_opt)

print(f"\n{'═'*68}")
print("  FINAL RESULTS")
print(f"{'═'*68}")
print(f"  Train Accuracy      : {train_acc*100:.2f}%")
print(f"  Test  Accuracy      : {test_acc*100:.2f}%  (threshold=0.50)")
print(f"  Test  Accuracy*     : {test_acc_opt*100:.2f}%  (threshold={best_thr:.2f}, optimised)")
print(f"  ROC-AUC Score       : {roc_auc:.4f}")
print(f"  Average Precision   : {avg_prec:.4f}")
print(f"  F1 Score (test)     : {test_f1:.2f}%\n")
print("  Classification Report:")
print(classification_report(y_test, y_pred,
      target_names=['Retained', 'Churned'], digits=4))


════════════════════════════════════════════════════════════════════
  FINAL RESULTS
════════════════════════════════════════════════════════════════════
  Train Accuracy      : 99.98%
  Test  Accuracy      : 96.59%  (threshold=0.50)
  Test  Accuracy*     : 97.04%  (threshold=0.25, optimised)
  ROC-AUC Score       : 0.9899
  Average Precision   : 0.9621
  F1 Score (test)     : 88.74%

  Classification Report:
              precision    recall  f1-score   support

    Retained     0.9695    0.9906    0.9799      1701
     Churned     0.9444    0.8369    0.8874       325

    accuracy                         0.9659      2026
   macro avg     0.9570    0.9138    0.9337      2026
weighted avg     0.9655    0.9659    0.9651      2026



In [10]:
BG     = '#080c14'
CARD   = '#111827'
BORDER = '#1e2d40'
ACCENT = '#3b82f6'
GREEN  = '#10b981'
RED    = '#ef4444'
ORANGE = '#f59e0b'
YELLOW = '#fde047'
PURPLE = '#8b5cf6'
CYAN   = '#06b6d4'
PINK   = '#ec4899'
TEXT   = '#f1f5f9'
GRAY   = '#64748b'
LGRAY  = '#94a3b8'

fig = plt.figure(figsize=(24, 20), facecolor=BG)
fig.patch.set_facecolor(BG)
gs = gridspec.GridSpec(4, 4, figure=fig,
                       hspace=0.52, wspace=0.38,
                       left=0.04, right=0.97,
                       top=0.92, bottom=0.05)

def ax_style(ax, title=''):
    ax.set_facecolor(CARD)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)
        sp.set_linewidth(0.8)
    ax.tick_params(colors=LGRAY, labelsize=8.5)
    ax.xaxis.label.set_color(LGRAY)
    ax.yaxis.label.set_color(LGRAY)
    if title:
        ax.set_title(title, color=TEXT, fontsize=10.5,
                     fontweight='bold', pad=9, loc='left')

# ── HEADER ────────────────────────────────────────────────────────────────────
fig.text(0.5, 0.96,
         '💳  Credit Card Customer Churn — ML Prediction Dashboard',
         ha='center', color=TEXT, fontsize=20, fontweight='bold',
         fontfamily='monospace')
fig.text(0.5, 0.935,
         f'Stacking Ensemble: HGB×2 + RandomForest + ExtraTrees + MLP + GradBoost  '
         f'│  Test Acc: {test_acc*100:.2f}%  │  ROC-AUC: {roc_auc:.4f}  '
         f'│  Avg Precision: {avg_prec:.4f}',
         ha='center', color=GRAY, fontsize=10.5)

# ── [R0,C0] KPI CARDS ────────────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0, 0])
ax0.set_facecolor(BG); ax0.axis('off')
ax0.set_title('Performance Metrics', color=TEXT, fontsize=10.5,
              fontweight='bold', pad=9, loc='left')
kpis = [
    ('Train Accuracy', f"{train_acc*100:.2f}%",   ACCENT,  '↑ Learned well'),
    ('Test Accuracy',  f"{test_acc*100:.2f}%",    GREEN,   '↑ Hits 95% target'),
    ('ROC-AUC Score',  f"{roc_auc:.4f}",          ORANGE,  '↑ Excellent rank'),
    ('F1-Score',       f"{test_f1:.2f}%",          PURPLE,  '↑ Balanced perf'),
]
for i, (lbl, val, col, sub) in enumerate(kpis):
    y0 = 0.86 - i * 0.22
    ax0.add_patch(mpatches.FancyBboxPatch(
        (0.03, y0-0.07), 0.94, 0.19,
        boxstyle="round,pad=0.015",
        facecolor=CARD, edgecolor=col, linewidth=1.8,
        transform=ax0.transAxes, zorder=2))
    ax0.text(0.5,  y0+0.06, val,  ha='center', va='center',
             color=col, fontsize=15, fontweight='bold',
             transform=ax0.transAxes, zorder=3)
    ax0.text(0.5,  y0-0.01, lbl,  ha='center', va='center',
             color=LGRAY, fontsize=8, transform=ax0.transAxes, zorder=3)
    ax0.text(0.5,  y0-0.05, sub,  ha='center', va='center',
             color=GRAY, fontsize=7.5, transform=ax0.transAxes, zorder=3)

# ── [R0,C1] CONFUSION MATRIX ─────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 1])
ax_style(ax1, 'Confusion Matrix')
cm = confusion_matrix(y_test, y_pred)
cmap_cm = LinearSegmentedColormap.from_list('cb', [CARD, ACCENT])
ax1.imshow(cm, cmap=cmap_cm, aspect='auto')
labels = [['TN\n(Correct\nRetained)', 'FP\n(False\nAlarm)'],
          ['FN\n(Missed\nChurn)', 'TP\n(Correct\nChurn)']]
for i in range(2):
    for j in range(2):
        clr = TEXT if cm[i,j] < cm.max()*0.55 else BG
        ax1.text(j, i, f'{cm[i,j]:,}\n{labels[i][j]}',
                 ha='center', va='center', color=clr,
                 fontsize=9, fontweight='bold')
ax1.set_xticks([0,1]); ax1.set_yticks([0,1])
ax1.set_xticklabels(['Retained','Churned'], color=LGRAY, fontsize=9)
ax1.set_yticklabels(['Retained','Churned'], color=LGRAY, fontsize=9)
ax1.set_xlabel('Predicted', color=LGRAY)
ax1.set_ylabel('Actual',    color=LGRAY)

# ── [R0,C2] ROC CURVE ────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax_style(ax2, 'ROC Curve (All Models)')
roc_colors = [ACCENT, CYAN, GREEN, ORANGE, PINK, YELLOW]
for (name, mdl), col in zip(models.items(), roc_colors):
    pp2 = mdl.predict_proba(X_te_sc)[:,1]
    fp2, tp2, _ = roc_curve(y_test, pp2)
    ax2.plot(fp2, tp2, color=col, lw=1.2, alpha=0.7,
             label=f'{name[:10]} ({ind_aucs[name]:.3f})')
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax2.plot(fpr, tpr, color=RED, lw=2.5,
         label=f'Stacking ({roc_auc:.4f})', zorder=5)
ax2.fill_between(fpr, tpr, alpha=0.08, color=RED)
ax2.plot([0,1],[0,1],'--', color=GRAY, lw=1)
ax2.set_xlim([0,1]); ax2.set_ylim([0,1.02])
ax2.set_xlabel('False Positive Rate', color=LGRAY)
ax2.set_ylabel('True Positive Rate',  color=LGRAY)
ax2.legend(fontsize=7, facecolor=CARD, edgecolor=BORDER,
           labelcolor=LGRAY, loc='lower right')
ax2.grid(True, alpha=0.08, color=GRAY)

# ── [R0,C3] PRECISION-RECALL CURVE ──────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 3])
ax_style(ax3, 'Precision-Recall Curve')
prec_c, rec_c, _ = precision_recall_curve(y_test, y_prob)
ax3.plot(rec_c, prec_c, color=GREEN, lw=2.5,
         label=f'Stacking AP={avg_prec:.4f}')
ax3.fill_between(rec_c, prec_c, alpha=0.1, color=GREEN)
baseline = y_test.mean()
ax3.axhline(baseline, color=GRAY, lw=1, ls='--',
            label=f'Baseline={baseline:.2f}')
ax3.set_xlabel('Recall',    color=LGRAY)
ax3.set_ylabel('Precision', color=LGRAY)
ax3.set_xlim([0,1]); ax3.set_ylim([0,1.05])
ax3.legend(fontsize=9, facecolor=CARD, edgecolor=BORDER, labelcolor=LGRAY)
ax3.grid(True, alpha=0.08, color=GRAY)

# ── [R1,C0:2] MODEL ACCURACY COMPARISON ──────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0:2])
ax_style(ax4, 'Model Accuracy Comparison (Test Set %)')
all_names = list(ind_accs.keys()) + ['Stacking\nEnsemble ★']
all_accs  = list(ind_accs.values()) + [test_acc*100]
bar_cols  = [ACCENT]*len(ind_accs) + [GREEN]
x_pos     = np.arange(len(all_names))
bars = ax4.bar(x_pos, all_accs, color=bar_cols,
               edgecolor=BORDER, linewidth=0.8, width=0.6, zorder=3)
ax4.set_ylim([70, 102])
for bar, val in zip(bars, all_accs):
    col = GREEN if val >= 95 else (ORANGE if val >= 91 else RED)
    ax4.text(bar.get_x()+bar.get_width()/2,
             bar.get_height()+0.3,
             f'{val:.2f}%', ha='center', va='bottom',
             color=col, fontweight='bold', fontsize=10)
ax4.set_xticks(x_pos)
ax4.set_xticklabels(all_names, color=LGRAY, fontsize=8.5)
ax4.set_ylabel('Test Accuracy (%)', color=LGRAY)
ax4.legend(fontsize=9, facecolor=CARD, edgecolor=BORDER, labelcolor=LGRAY)
ax4.grid(True, alpha=0.08, axis='y', color=GRAY, zorder=0)

# ── [R1,C2:4] FEATURE IMPORTANCES ────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2:4])
ax_style(ax5, 'Top 18 Feature Importances (Random Forest)')
fi = pd.Series(rf.feature_importances_,
               index=feature_names).sort_values(ascending=True).tail(18)
cmap_fi = plt.cm.get_cmap('cool')
colors_fi = [cmap_fi(i/len(fi)) for i in range(len(fi))]
bars_fi = ax5.barh(fi.index, fi.values, color=colors_fi,
                   edgecolor=BORDER, linewidth=0.4, height=0.7)
for bar, val in zip(bars_fi, fi.values):
    ax5.text(val+0.0003, bar.get_y()+bar.get_height()/2,
             f'{val:.4f}', va='center', color=LGRAY, fontsize=8)
ax5.set_xlabel('Importance Score', color=LGRAY)
ax5.tick_params(axis='y', labelsize=8)
ax5.grid(True, alpha=0.08, axis='x', color=GRAY)

# ── [R2,C0] TOTAL TRANS CT DISTRIBUTION ──────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 0])
ax_style(ax6, 'Total Transactions by Churn')
df_orig = df.copy()
ax6.hist(df_orig[df_orig['Churn']==0]['Total_Trans_Ct'],
         bins=35, color=GREEN,  alpha=0.65, density=True, label='Retained')
ax6.hist(df_orig[df_orig['Churn']==1]['Total_Trans_Ct'],
         bins=35, color=RED,    alpha=0.70, density=True, label='Churned')
ax6.set_xlabel('Total Transaction Count', color=LGRAY)
ax6.set_ylabel('Density', color=LGRAY)
ax6.legend(fontsize=8.5, facecolor=CARD, edgecolor=BORDER, labelcolor=LGRAY)
ax6.grid(True, alpha=0.08, color=GRAY)

# ── [R2,C1] UTILIZATION RATIO DISTRIBUTION ───────────────────────────────────
ax7 = fig.add_subplot(gs[2, 1])
ax_style(ax7, 'Utilization Ratio by Churn')
ax7.hist(df_orig[df_orig['Churn']==0]['Avg_Utilization_Ratio'],
         bins=30, color=CYAN,  alpha=0.65, density=True, label='Retained')
ax7.hist(df_orig[df_orig['Churn']==1]['Avg_Utilization_Ratio'],
         bins=30, color=PINK,  alpha=0.70, density=True, label='Churned')
ax7.set_xlabel('Avg Utilization Ratio', color=LGRAY)
ax7.set_ylabel('Density', color=LGRAY)
ax7.legend(fontsize=8.5, facecolor=CARD, edgecolor=BORDER, labelcolor=LGRAY)
ax7.grid(True, alpha=0.08, color=GRAY)

# ── [R2,C2] CHURN BY INCOME CATEGORY ─────────────────────────────────────────
ax8 = fig.add_subplot(gs[2, 2])
ax_style(ax8, 'Churn Rate by Income Category')
# Income_Category is already label-encoded; use original if available
if 'Income_Category' in df_orig.columns:
    ic_churn = df_orig.groupby('Income_Category')['Churn'].mean() * 100
    inc_cols  = [GREEN if v < 15 else (ORANGE if v < 20 else RED)
                 for v in ic_churn.values]
    bars_ic = ax8.bar(range(len(ic_churn)), ic_churn.values,
                      color=inc_cols, edgecolor=BORDER, width=0.55)
    ax8.set_xticks(range(len(ic_churn)))
    ax8.set_xticklabels([str(i) for i in ic_churn.index],
                        color=LGRAY, fontsize=8, rotation=15)
    for bar, val in zip(bars_ic, ic_churn.values):
        ax8.text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.2,
                 f'{val:.1f}%', ha='center', color=TEXT,
                 fontweight='bold', fontsize=9)
    ax8.set_ylabel('Churn Rate (%)', color=LGRAY)
    ax8.set_ylim([0, 35])
ax8.grid(True, alpha=0.08, axis='y', color=GRAY)

# ── [R2,C3] CONTACTS COUNT VS CHURN RATE ─────────────────────────────────────
ax9 = fig.add_subplot(gs[2, 3])
ax_style(ax9, 'Churn Rate by Contacts Count')
cc_churn = df_orig.groupby('Contacts_Count_12_mon')['Churn'].mean() * 100
c_cols   = [GREEN if v < 15 else (ORANGE if v < 25 else RED)
            for v in cc_churn.values]
bars_cc  = ax9.bar(cc_churn.index, cc_churn.values,
                   color=c_cols, edgecolor=BORDER, width=0.6)
for bar, val in zip(bars_cc, cc_churn.values):
    ax9.text(bar.get_x()+bar.get_width()/2,
             bar.get_height()+0.4,
             f'{val:.1f}%', ha='center', color=TEXT,
             fontweight='bold', fontsize=9)
ax9.set_xlabel('Contacts in Last 12 Months', color=LGRAY)
ax9.set_ylabel('Churn Rate (%)',              color=LGRAY)
ax9.set_ylim([0, 55])
ax9.grid(True, alpha=0.08, axis='y', color=GRAY)

# ── [R3,C0] PREDICTED PROBABILITY DIST ───────────────────────────────────────
ax10 = fig.add_subplot(gs[3, 0])
ax_style(ax10, 'Predicted Churn Probability')
ax10.hist(y_prob[y_test==0], bins=45, color=GREEN, alpha=0.65,
          density=True, label='Retained')
ax10.hist(y_prob[y_test==1], bins=45, color=RED,   alpha=0.70,
          density=True, label='Churned')
ax10.axvline(0.5,      color=YELLOW, lw=1.8, ls='--', label='Threshold=0.5')
ax10.axvline(best_thr, color=CYAN,   lw=1.5, ls=':',
             label=f'Optimal={best_thr:.2f}')
ax10.set_xlabel('Predicted Probability', color=LGRAY)
ax10.set_ylabel('Density', color=LGRAY)
ax10.legend(fontsize=8, facecolor=CARD, edgecolor=BORDER, labelcolor=LGRAY)
ax10.grid(True, alpha=0.08, color=GRAY)

# ── [R3,C1] TRANS AMT vs TRANS CT scatter ────────────────────────────────────
ax11 = fig.add_subplot(gs[3, 1])
ax_style(ax11, 'Trans Amount vs Count (by Churn)')
sample_idx = np.random.choice(len(df_orig), min(3000, len(df_orig)), replace=False)
df_s = df_orig.iloc[sample_idx]
ax11.scatter(df_s[df_s['Churn']==0]['Total_Trans_Ct'],
             df_s[df_s['Churn']==0]['Total_Trans_Amt']/1000,
             c=GREEN, alpha=0.25, s=6, label='Retained')
ax11.scatter(df_s[df_s['Churn']==1]['Total_Trans_Ct'],
             df_s[df_s['Churn']==1]['Total_Trans_Amt']/1000,
             c=RED,   alpha=0.5,  s=8, label='Churned')
ax11.set_xlabel('Total Transaction Count', color=LGRAY)
ax11.set_ylabel('Total Trans Amount (K)',  color=LGRAY)
ax11.legend(fontsize=8.5, facecolor=CARD, edgecolor=BORDER,
            labelcolor=LGRAY, markerscale=3)
ax11.grid(True, alpha=0.08, color=GRAY)

# ── [R3,C2] MODEL F1 COMPARISON ──────────────────────────────────────────────
ax12 = fig.add_subplot(gs[3, 2])
ax_style(ax12, 'Model F1-Score Comparison (%)')
all_f1s  = list(ind_f1s.values()) + [test_f1]
f1_cols  = [PURPLE]*len(ind_f1s) + [ORANGE]
bars_f1  = ax12.bar(x_pos, all_f1s, color=f1_cols,
                    edgecolor=BORDER, width=0.6, zorder=3)
ax12.set_ylim([50, 100])
for bar, val in zip(bars_f1, all_f1s):
    ax12.text(bar.get_x()+bar.get_width()/2,
              bar.get_height()+0.5,
              f'{val:.1f}%', ha='center', color=TEXT,
              fontweight='bold', fontsize=9)
ax12.set_xticks(x_pos)
ax12.set_xticklabels(all_names, color=LGRAY, fontsize=8, rotation=15)
ax12.set_ylabel('F1 Score (%)', color=LGRAY)
ax12.grid(True, alpha=0.08, axis='y', color=GRAY, zorder=0)

# ── [R3,C3] MONTHS INACTIVE BY CHURN ─────────────────────────────────────────
ax13 = fig.add_subplot(gs[3, 3])
ax_style(ax13, 'Months Inactive vs Churn Rate')
mi_churn = df_orig.groupby('Months_Inactive_12_mon')['Churn'].mean() * 100
m_cols   = [GREEN if v < 15 else (ORANGE if v < 25 else RED)
            for v in mi_churn.values]
bars_mi  = ax13.bar(mi_churn.index, mi_churn.values,
                    color=m_cols, edgecolor=BORDER, width=0.6)
for bar, val in zip(bars_mi, mi_churn.values):
    ax13.text(bar.get_x()+bar.get_width()/2,
              bar.get_height()+0.4,
              f'{val:.1f}%', ha='center', color=TEXT,
              fontweight='bold', fontsize=9)
ax13.set_xlabel('Months Inactive (Last 12)', color=LGRAY)
ax13.set_ylabel('Churn Rate (%)',             color=LGRAY)
ax13.grid(True, alpha=0.08, axis='y', color=GRAY)

plt.savefig(OUTPUT_IMG, dpi=150, bbox_inches='tight', facecolor=BG)
plt.close()
print(f"\n  ✓ Dashboard saved → {OUTPUT_IMG}")


  ✓ Dashboard saved → bankchurners_dashboard.png


In [11]:
print("\n" + "═"*68)
print("  FINAL SUMMARY")
print("═"*68)
print(f"  Dataset        : Credit Card BankChurners (Kaggle - sakshigoyal7)")
print(f"  Algorithm      : Stacking Ensemble")
print(f"  Base models    : HGB×2 + RandomForest + ExtraTrees + MLP + GradBoost")
print(f"  Meta-learner   : Logistic Regression (C=8.0)")
print(f"  Features       : {X.shape[1]} total (original + 18 engineered)")
print(f"  Balancing      : Targeted oversampling (minority → 65% of majority)")
print(f"  Scaling        : StandardScaler")
print(f"  ──────────────────────────────────────────────────────────────")
print(f"  Train Accuracy : {train_acc*100:.2f}%")
print(f"  Test  Accuracy : {test_acc*100:.2f}%  ← Target: 95%")
print(f"  ROC-AUC Score  : {roc_auc:.4f}")
print(f"  F1-Score       : {test_f1:.2f}%")
print(f"  Avg Precision  : {avg_prec:.4f}")
print("═"*68)
print()
print("  WHY THIS DATASET ACHIEVES 95%+:")
print("  ✓ Total_Trans_Ct  — most predictive behavioral signal")
print("  ✓ Total_Trans_Amt — spending amount separates classes clearly")
print("  ✓ Total_Ct_Chng   — change in activity is a churn signal")
print("  ✓ Avg_Utilization — low utilizers churn more")
print("  ✓ Months_Inactive — inactivity directly signals churn risk")
print("═"*68 + "\n")


════════════════════════════════════════════════════════════════════
  FINAL SUMMARY
════════════════════════════════════════════════════════════════════
  Dataset        : Credit Card BankChurners (Kaggle - sakshigoyal7)
  Algorithm      : Stacking Ensemble
  Base models    : HGB×2 + RandomForest + ExtraTrees + MLP + GradBoost
  Meta-learner   : Logistic Regression (C=8.0)
  Features       : 37 total (original + 18 engineered)
  Balancing      : Targeted oversampling (minority → 65% of majority)
  Scaling        : StandardScaler
  ──────────────────────────────────────────────────────────────
  Train Accuracy : 99.98%
  Test  Accuracy : 96.59%  ← Target: 95%
  ROC-AUC Score  : 0.9899
  F1-Score       : 88.74%
  Avg Precision  : 0.9621
════════════════════════════════════════════════════════════════════

  WHY THIS DATASET ACHIEVES 95%+:
  ✓ Total_Trans_Ct  — most predictive behavioral signal
  ✓ Total_Trans_Amt — spending amount separates classes clearly
  ✓ Total_Ct_Chng   — change 